# ETL Features - Fase 1
## Sistema de Recomendaciones E-commerce
## Responsable: Gama
## Semana 2 - Big Data Project

Lee las interacciones crudas desde MinIO, aplica decay temporal
y genera los features listos para el modelo ALS.

In [1]:
try:
    spark.stop()
    print("✅ SparkSession detenida")
except:
    print("No había sesión activa")

No había sesión activa


In [2]:
import os

jars = [
    "/opt/spark/jars/hadoop-aws-3.3.4.jar",
    "/opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar"
]

for jar in jars:
    existe = os.path.exists(jar)
    size = os.path.getsize(jar) / (1024*1024) if existe else 0
    print(f"{'✅' if existe else '❌'} {jar} ({size:.1f} MB)")

✅ /opt/spark/jars/hadoop-aws-3.3.4.jar (0.9 MB)
✅ /opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar (267.6 MB)


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

In [4]:
MASTER_URL       = "spark://10.228.0.41:7077"
MINIO_ENDPOINT   = "http://10.228.0.41:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin12345"

MINIO_INTERACTIONS = "s3a://raw/interactions_run_date=2026-05-10/"
MINIO_CATALOG      = "s3a://raw/catalog_run_date=2026-05-10/"
MINIO_OUTPUT       = "s3a://features/etl_run_date=2026-05-13/"

JARS = ",".join([
    "/opt/spark/jars/hadoop-aws-3.3.4.jar",
    "/opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar"
])

In [5]:
spark = SparkSession.builder \
    .appName("ETL_Features_Ecommerce") \
    .master(MASTER_URL) \
    .config("spark.jars", JARS) \
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl",              "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.memory",   "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} — Master: {spark.sparkContext.master}")

26/05/13 16:27:04 WARN Utils: Your hostname, MacBook-Air-de-Gamaliel.local resolves to a loopback address: 127.0.0.1; using 10.228.3.202 instead (on interface en0)
26/05/13 16:27:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/05/13 16:27:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Spark 3.5.8 — Master: spark://10.228.0.41:7077


In [6]:
print("📖 Leyendo interacciones desde MinIO...")
interactions_raw = spark.read.csv(
    MINIO_INTERACTIONS, header=True, inferSchema=True
)

print("📖 Leyendo catálogo desde MinIO...")
catalog_raw = spark.read.csv(
    MINIO_CATALOG, header=True, inferSchema=True
)

# Solo muestra el schema, sin count() todavía
print("\nSchema interacciones:")
interactions_raw.printSchema()
print("\nSchema catálogo:")
catalog_raw.printSchema()

# Muestra 5 filas rápido (no escanea todo el dataset)
print("\nMuestra interacciones:")
interactions_raw.show(5)
print("\nMuestra catálogo:")
catalog_raw.show(5)

📖 Leyendo interacciones desde MinIO...


26/05/13 16:27:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/13 16:29:13 ERROR TaskSchedulerImpl: Lost executor 1 on 100.78.9.93: Unable to create executor due to null
26/05/13 16:29:13 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/jars/aws-java-sdk-bundle-1.12.262.jar,byteCount=280645251,body=FileSegmentManagedBuffer[file=/opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar,offset=0,length=280645251]] to /10.228.1.120:61734; closing connection
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel.close(ChannelPromise)(Unknown Source)
26/05/13 16:29:13 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/jars/aws-java-sdk-bundle-1.12.262.jar,byteCount=280645251,body=FileSegmentManagedBuffer[file=/opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar,offset=0,length=280645251]] to /10.228.1.127:51482; closing connection
io.n

📖 Leyendo catálogo desde MinIO...



Schema interacciones:
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)


Schema catálogo:
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- stock: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- category: string (nullable = true)


Muestra interacciones:
+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|   3949|   6.0|
|     1|   7938|   1.0|
|     2|   6287|   1.0|
|     3|   3300|   3.0|
|     3|   3980|   3.0|
+------+-------+------+
only showing top 5 rows


Muestra catálogo:
+-------+--------------------+--------------------+---------+-----+-----+--------------------+
|movieId|               title|              genres|is_active|stock|price|            category|
+-------+--------------------+--------------------+---------+-----+-----+-----------

26/05/13 16:31:31 ERROR TaskSchedulerImpl: Lost executor 3 on 10.228.1.127: Unable to create executor due to null
26/05/13 16:31:31 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/jars/aws-java-sdk-bundle-1.12.262.jar,byteCount=280645251,body=FileSegmentManagedBuffer[file=/opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar,offset=0,length=280645251]] to /10.228.1.127:51416; closing connection
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel.close(ChannelPromise)(Unknown Source)
26/05/13 16:31:31 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/jars/aws-java-sdk-bundle-1.12.262.jar,byteCount=280645251,body=FileSegmentManagedBuffer[file=/opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar,offset=0,length=280645251]] to /10.228.1.120:49822; closing connection
java.io.IOException: Broken pipe
	at java.base/sun.nio.ch.FileDispatcherImpl.write0(Native Method)
	at java.base/sun.nio.ch.SocketDispatcher.write

In [7]:
from pyspark.sql import functions as F
import datetime

print("⚙️ Aplicando decay temporal y construyendo features...")

HOY = datetime.date.today().isoformat()

interactions = interactions_raw \
    .withColumn("score_raw", F.col("rating") / 10.0) \
    .withColumn("run_date", F.lit(HOY).cast("date")) \
    .withColumn("days_old", F.lit(0)) \
    .withColumn("decay", F.exp(F.col("days_old") * F.lit(-0.01))) \
    .withColumn("score", F.round(F.col("score_raw") * F.col("decay"), 6))

features = interactions.join(
    catalog_raw.select("movieId", "price", "stock", "is_active", "category"),
    on="movieId",
    how="inner"
) \
.filter(F.col("is_active") == True) \
.filter(F.col("stock") > 0) \
.select(
    F.col("userId").alias("user_id"),
    F.col("movieId").alias("item_id"),
    F.col("score"),
    F.col("price"),
    F.col("stock"),
    F.col("category"),
    F.col("run_date")
)

print("✅ Features construidos")
features.show(5)

⚙️ Aplicando decay temporal y construyendo features...
✅ Features construidos


+-------+-------+-----+-----+-----+--------------------+----------+
|user_id|item_id|score|price|stock|            category|  run_date|
+-------+-------+-----+-----+-----+--------------------+----------+
|      1|   3949|  0.6|29.27|   11|               Drama|2026-05-13|
|      1|   7938|  0.1|54.78|   89|               Drama|2026-05-13|
|      2|   6287|  0.1|26.85|   53|              Comedy|2026-05-13|
|      3|   3300|  0.3|19.12|   61|Horror|Sci-Fi|Thr...|2026-05-13|
|      3|   3980|  0.3|14.43|   98|               Drama|2026-05-13|
+-------+-------+-----+-----+-----+--------------------+----------+
only showing top 5 rows



In [13]:
spark.stop()

spark = SparkSession.builder \
    .appName("ETL_Features_Local_Test") \
    .master("local[4]") \
    .config("spark.jars", JARS) \
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl",              "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory",   "4g") \
    .getOrCreate()

print(f"✅ Spark {spark.version} — Master: {spark.sparkContext.master}")

✅ Spark 3.5.1 — Master: local[4]


In [8]:
print("💾 Guardando features en MinIO...")

features.write \
    .mode("overwrite") \
    .partitionBy("run_date") \
    .csv(MINIO_OUTPUT, header=True)

print(f"✅ Features guardados en {MINIO_OUTPUT}")

💾 Guardando features en MinIO...


26/05/13 16:32:56 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

✅ Features guardados en s3a://features/etl_run_date=2026-05-13/
